# Route Resilience — Train the occlusion-robust road segmenter (Kaggle)

Trains a **D-LinkNet** (LinkNet + ResNet34) on **DeepGlobe** in two variants:
- **baseline** — BCE+Dice on clean chips (fragments under occlusion).
- **robust** — clean **+** synthetically occluded chips **+** consistency loss (sees through occlusion).

### Before you run (Kaggle right-hand panel)
1. **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Settings → Internet → On** (needed to clone the repo + download ImageNet encoder weights).
3. **Add Input → Datasets →** search and add **`balraj98/deepglobe-road-extraction-dataset`**.

Then **Run All**. Checkpoints + the real metrics table land in `/kaggle/working/` (downloadable as notebook output).

In [ ]:
# 0) Environment check
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# 1) Get the project code (public repo) + install the few deps Kaggle lacks.
#    Kaggle already ships torch+CUDA / numpy / Pillow, so we DON'T reinstall torch.
import os
REPO = '/kaggle/working/route-resilience'
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Ritesh-Root/route-resilience.git {REPO}
!pip install -q segmentation-models-pytorch timm tqdm
%cd {REPO}
print('repo ready at', REPO)

In [ ]:
# 2) Knobs — tune these for your time budget.
N_IMAGES = 2000     # how many DeepGlobe train tiles to use (None = all ~6226). Start small, scale up.
VAL_FRAC = 0.1      # held-out fraction for validation / metrics
TILE     = 512      # chip size fed to the net
EPOCHS   = 12       # 10-15 gives a presentable IoU; raise once you've confirmed the run completes
BATCH    = 8        # T4/P100 16GB handles 8 at 512 with AMP; drop to 6 if you OOM on the robust run
print(dict(N_IMAGES=N_IMAGES, VAL_FRAC=VAL_FRAC, TILE=TILE, EPOCHS=EPOCHS, BATCH=BATCH))

In [ ]:
# 3) Build 512x512 chips in the layout ml/train/train.py expects:
#       chips/<split>/images/<id>.png  +  chips/<split>/masks/<id>.png  (same basename)
import os, glob, random
from PIL import Image

# Locate the DeepGlobe train folder (has <id>_sat.jpg + <id>_mask.png).
cands = glob.glob('/kaggle/input/**/train/*_sat.jpg', recursive=True)
assert cands, 'DeepGlobe not found — did you add balraj98/deepglobe-road-extraction-dataset as Input?'
SRC = os.path.dirname(cands[0])
sats = sorted(glob.glob(os.path.join(SRC, '*_sat.jpg')))
print('found', len(sats), 'labelled tiles in', SRC)

random.seed(1337); random.shuffle(sats)
if N_IMAGES: sats = sats[:N_IMAGES]
n_val = max(1, int(len(sats) * VAL_FRAC))
splits = {'val': sats[:n_val], 'train': sats[n_val:]}

CHIPS = '/kaggle/working/chips'
for split, files in splits.items():
    idir = f'{CHIPS}/{split}/images'; mdir = f'{CHIPS}/{split}/masks'
    os.makedirs(idir, exist_ok=True); os.makedirs(mdir, exist_ok=True)
    for sat in files:
        mask = sat.replace('_sat.jpg', '_mask.png')
        if not os.path.isfile(mask): continue
        base = os.path.basename(sat).replace('_sat.jpg', '') + '.png'
        Image.open(sat).convert('RGB').resize((TILE, TILE), Image.BILINEAR).save(f'{idir}/{base}')
        Image.open(mask).convert('L').resize((TILE, TILE), Image.NEAREST).save(f'{mdir}/{base}')
    print(f'{split}: {len(os.listdir(idir))} chips')
print('chips ready at', CHIPS)

In [ ]:
# 4) Smoke test — verifies torch + smp + the loss/model imports and runs one tiny step.
#    If this prints a final loss, the long runs below will work.
!python -m ml.train.train --self-test

In [ ]:
# 5) Train the BASELINE (clean-only). ~1-2 hrs depending on N_IMAGES/EPOCHS.
!python -m ml.train.train --variant baseline \
    --data /kaggle/working/chips \
    --out  /kaggle/working/checkpoints/baseline.pt \
    --epochs {EPOCHS} --batch-size {BATCH} --accum-steps 1 \
    --tile {TILE} --num-workers 2

In [ ]:
# 6) Train the ROBUST model (clean + occluded + consistency loss — auto-enabled by --variant robust).
#    Robust forwards 2 views, so it uses a bit more VRAM; drop --batch-size to 6 if you OOM.
!python -m ml.train.train --variant robust \
    --data /kaggle/working/chips \
    --out  /kaggle/working/checkpoints/robust.pt \
    --epochs {EPOCHS} --batch-size {BATCH} --accum-steps 1 \
    --tile {TILE} --num-workers 2

In [ ]:
# 7) Evaluate both models on CLEAN and OCCLUDED validation — these are the REAL numbers
#    to replace the placeholder constants in backend/app/main.py:_METRIC_TABLE.
import sys, glob, os, numpy as np, torch
sys.path.insert(0, '/kaggle/working/route-resilience')  # make ml.* importable regardless of cwd
from PIL import Image
from ml.models.dlinknet import build_dlinknet
from ml.train.train import _load_occluder

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
occlude = _load_occluder()
val_imgs = sorted(glob.glob('/kaggle/working/chips/val/images/*.png'))

def load_model(path):
    m = build_dlinknet(encoder_weights=None).to(DEV).eval()
    ck = torch.load(path, map_location=DEV)
    m.load_state_dict(ck['model'] if 'model' in ck else ck)
    return m

def metrics(model, occluded=False):
    inter=union=tp=fn=dice_n=dice_d=0.0
    with torch.no_grad():
        for ip in val_imgs:
            mp = ip.replace('/images/', '/masks/')
            img = np.asarray(Image.open(ip).convert('RGB'), dtype='uint8')
            if occluded: img = occlude(img, seed=hash(ip) % 100000)
            gt = (np.asarray(Image.open(mp).convert('L')) > 127).astype('float32')
            x = torch.from_numpy(img.transpose(2,0,1).copy()).float().div(255.).unsqueeze(0).to(DEV)
            pr = (torch.sigmoid(model(x))[0,0].cpu().numpy() > 0.5).astype('float32')
            inter += (pr*gt).sum(); union += ((pr+gt) > 0).sum()
            tp += (pr*gt).sum(); fn += ((1-pr)*gt).sum()
            dice_n += 2*(pr*gt).sum(); dice_d += pr.sum()+gt.sum()
    iou = inter/(union+1e-6); dice = dice_n/(dice_d+1e-6); recall = tp/(tp+fn+1e-6)
    return dict(IoU=round(float(iou),3), Dice=round(float(dice),3), Recall=round(float(recall),3))

rows = []
for name in ('baseline', 'robust'):
    p = f'/kaggle/working/checkpoints/{name}.pt'
    if not os.path.isfile(p):
        # fall back to a best-checkpoint sibling if the loop saved one
        alt = glob.glob(f'/kaggle/working/checkpoints/{name}*best*.pt')
        p = alt[0] if alt else p
    m = load_model(p)
    rows.append((name, 'clean',    metrics(m, occluded=False)))
    rows.append((name, 'occluded', metrics(m, occluded=True)))

print(f"{'model':10}{'input':10}{'IoU':>8}{'Dice':>8}{'Recall':>8}")
for name, inp, d in rows:
    print(f"{name:10}{inp:10}{d['IoU']:>8}{d['Dice']:>8}{d['Recall']:>8}")
print('\n^ Occlusion-Recall = the Recall row at input=occluded. Robust should beat baseline there — that IS the story.')

In [ ]:
# 8) Visual proof — run the trained ROBUST model on a few clean + occluded val tiles
#    and save side-by-side overlays (predicted roads in orange). Great for slides.
import matplotlib.pyplot as plt
robust_ckpt = '/kaggle/working/checkpoints/robust.pt'
assert os.path.isfile(robust_ckpt), 'train the robust model first (cell 6)'
model = load_model(robust_ckpt)
samples = sorted(glob.glob('/kaggle/working/chips/val/images/*.png'))[:4]
os.makedirs('/kaggle/working/overlays', exist_ok=True)

def predict_overlay(img):
    x = torch.from_numpy(img.transpose(2,0,1).copy()).float().div(255.).unsqueeze(0).to(DEV)
    with torch.no_grad():
        pr = (torch.sigmoid(model(x))[0,0].cpu().numpy() > 0.5)
    ov = img.copy(); ov[pr] = [255, 90, 0]
    return ov

fig, ax = plt.subplots(len(samples), 3, figsize=(11, 3.4*len(samples)))
for r, ip in enumerate(samples):
    clean = np.asarray(Image.open(ip).convert('RGB'), dtype='uint8')
    occ = occlude(clean, seed=r)
    for c, (title, im) in enumerate([('input (clean)', clean),
                                     ('robust pred on CLEAN', predict_overlay(clean)),
                                     ('robust pred on OCCLUDED', predict_overlay(occ))]):
        ax[r, c].imshow(im); ax[r, c].set_title(title, fontsize=9); ax[r, c].axis('off')
    Image.fromarray(np.concatenate([clean, predict_overlay(clean)], 1)).save(
        f'/kaggle/working/overlays/{os.path.basename(ip)}')
plt.tight_layout(); plt.savefig('/kaggle/working/overlays/_grid.png', dpi=110); plt.show()
print('overlays saved to /kaggle/working/overlays — the robust model should still trace roads under occlusion.')

## From trained model → the live map (two paths)

The backend already serves **real OSM Bengaluru roads** (`backend/app/data/real_network.geojson`, via the Overpass fetcher) — so the dashboard map is real *today*, independent of training. The trained model adds two things:

**A. Real segmentation metrics (do this now).** Copy the IoU / Dice / Occlusion-Recall printed in cell 7 into `backend/app/main.py:_METRIC_TABLE`, replacing the placeholder constants. These become the honest KPI numbers.

**B. Model-extracted graph (optional, needs a georeferenced tile).** To replace the OSM graph with one your model extracts from imagery, you need a **georeferenced** tile (e.g. a Cartosat GeoTIFF with a lat/lng transform — provided during the hackathon). Then on a box with `scikit-image` installed:
```python
# pseudo-flow — see ml/integration/segmenter_hook.py
mask  = predict_mask(tile)                  # ml/inference/predict.py (the model you just trained)
graph = mask_to_graph(mask, geo_transform)  # ml/vectorize/* : skeletonize → graph → MST heal
geojson = graph_to_geojson(annotate(graph)) # backend schema (id, lng/lat, travelTimeSec, roadClass…)
# write to backend/app/data/real_network.geojson  → restart backend → map shows YOUR extraction
```
Without a geo-transform the graph is in pixel space (fine for proving extraction, not for the Leaflet map). The `segmenter_hook.py` seam is built so this is a drop-in: `network_factory` auto-serves whatever `real_network.geojson` contains, and criticality / gatekeepers / simulate are unchanged.

**Download from this notebook's Output tab:** `checkpoints/baseline.pt`, `checkpoints/robust.pt`, and `overlays/_grid.png`.

## Next steps after this notebook

1. **Download** `checkpoints/baseline.pt` and `checkpoints/robust.pt` from the notebook **Output** tab.
2. **Replace the placeholder metrics** in `backend/app/main.py` (`_METRIC_TABLE`) with the real IoU / Dice / Occlusion-Recall printed above.
3. **Wire the real graph**: run inference on a Bengaluru tile (`ml/inference/predict.py`) → `ml/vectorize/*` → heal → graph, then point `backend/app/services/network_factory.get_network()` at `ml/integration/segmenter_hook.py`. Everything downstream (criticality, gatekeepers, simulate, GeoJSON) is unchanged.

### Long runs
For a full-data run, set `N_IMAGES = None` and use **Save Version → Save & Run All (Commit)** so it runs in the background (up to ~9–12h). If a session is interrupted, re-run with `--resume /kaggle/working/checkpoints/<name>.pt` to continue from the last checkpoint.